In [ ]:
# Cài đặt các thư viện cần thiết
!pip install transformers datasets evaluate seqeval torch accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
import requests

# Link raw đến file .conll trên GitHub
train_url = "https://raw.githubusercontent.com/kodomotachi/NER/main/data/processed/train.conll"
valid_url = "https://raw.githubusercontent.com/kodomotachi/NER/main/data/processed/valid.conll"

def download_conll(url, save_path):
    response = requests.get(url)
    with open(save_path, 'wb') as f:
        f.write(response.content)

download_conll(train_url, "train.conll")
download_conll(valid_url, "valid.conll")

print("Tải dữ liệu thành công!")

Tải dữ liệu thành công!


In [ ]:
def read_conll(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        tokens = []
        ner_tags = []
        data = []

        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({"tokens": tokens, "ner_tags": ner_tags})
                    tokens = []
                    ner_tags = []
            else:
                # Dữ liệu CoNLL thường cách nhau bằng tab hoặc space
                parts = line.split()
                tokens.append(parts[0])
                ner_tags.append(parts[-1])

        # Thêm câu cuối cùng nếu file không kết thúc bằng dòng trống
        if tokens:
            data.append({"tokens": tokens, "ner_tags": ner_tags})

    return data

train_data = read_conll("train.conll")
valid_data = read_conll("valid.conll")

print(f"Số lượng câu trong train: {len(train_data)}")
print(f"Số lượng câu trong valid: {len(valid_data)}")
print(f"Ví dụ 1 câu train: {train_data[0]}")

Số lượng câu trong train: 2527
Số lượng câu trong valid: 316
Ví dụ 1 câu train: {'tokens': ['Invoice', 'is', 'made', 'by', 'me/', 'us', 'and', 'that', 'the', 'transaction', 'of', 'sales', 'covered', 'by', 'this', 'tax'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [ ]:
from transformers import AutoTokenizer

# Sử dụng bert-base-cased vì dữ liệu hóa đơn có nhiều tên riêng và mã viết hoa
model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Lấy tất cả các nhãn duy nhất từ dữ liệu train
unique_tags = set()
for item in train_data:
    for tag in item["ner_tags"]:
        unique_tags.add(tag)

unique_tags = sorted(list(unique_tags))
tag2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2tag = {idx: tag for tag, idx in tag2id.items()}

print("Label Mapping (tag2id):", tag2id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Label Mapping (tag2id): {'B-BUY_G': 0, 'B-BUY_N': 1, 'B-GSTL': 2, 'B-GT_AMT': 3, 'B-GT_AMTL': 4, 'B-INV_DL': 5, 'B-INV_DT': 6, 'B-INV_L': 7, 'B-INV_NO': 8, 'B-SUPP_G': 9, 'B-SUPP_N': 10, 'I-BUY_G': 11, 'I-BUY_N': 12, 'I-GSTL': 13, 'I-GT_AMT': 14, 'I-GT_AMTL': 15, 'I-INV_DL': 16, 'I-INV_DT': 17, 'I-INV_L': 18, 'I-INV_NO': 19, 'I-SUPP_G': 20, 'I-SUPP_N': 21, 'O': 22}


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map các subword về từ gốc
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Token đặc biệt như [CLS], [SEP] thì ignore
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Từ đầu tiên của từ gốc thì gán nhãn đúng
                label_ids.append(tag2id[label[word_idx]])
            else:
                # Các subword tiếp theo cũng ignore (hoặc có thể gán I- tag tùy chiến lược, ở đây dùng ignore cho đơn giản)
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Chuyển dữ liệu sang format của HuggingFace Dataset để xử lý nhanh
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

# Apply hàm align
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_valid = valid_dataset.map(tokenize_and_align_labels, batched=True)

print("Đã tokenize và align nhãn xong!")

Map:   0%|          | 0/2527 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Đã tokenize và align nhãn xong!


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(unique_tags),
    id2label=id2tag,
    label2id=tag2id
)

print(f"Đã tải model {model_checkpoint} với {len(unique_tags)} nhãn.")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã tải model roberta-base với 23 nhãn.


In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Bỏ đi các vị trí có label là -100 (ignore index)
    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./ner_invoice_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Bắt đầu huấn luyện
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.007887,0.973269,0.990111,0.981618
2,No log,0.005755,0.982864,0.992583,0.987700
3,No log,0.005703,0.985294,0.993820,0.989538
4,0.003878,0.005304,0.987715,0.993820,0.990758
5,0.003878,0.005440,0.986486,0.992583,0.989526


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=790, training_loss=0.0032447682905800733, metrics={'train_runtime': 1541.1582, 'train_samples_per_second': 8.198, 'train_steps_per_second': 0.513, 'total_flos': 3054213081954852.0, 'train_loss': 0.0032447682905800733, 'epoch': 5.0})

In [ ]:
# Đánh giá lại trên tập valid để thấy bảng kết quả chi tiết
results = trainer.evaluate()

print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"F1 Score:  {results['eval_f1']:.4f}")



--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
Precision: 0.9877
Recall:    0.9938
F1 Score:  0.9908
